# Project Template: Binary Classification with Evaluation

- Module band: 01-04
- Estimated runtime: 10-15 minutes for the starter notebook
- Estimated project time: 4-6 hours
- Prerequisites: logistic regression, confusion matrices, precision/recall, ROC-AUC
- Expected outputs: metric table, threshold comparison, confusion matrix, written interpretation


## Learning goals

- Fit a reproducible binary logistic-regression pipeline.
- Distinguish probabilities from thresholded class decisions.
- Compare multiple operating thresholds instead of accuracy alone.
- Write a short evidence-based justification for a preferred threshold.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 7
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target
dataset.target_names


## Step 1: Frame the classification task

Add a short response here:

- What does the positive class mean in this dataset?
- Why is logistic regression a good first classifier?
- Why can accuracy alone be misleading?


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=SEED,
)

print({
    "train_positive_rate": float(y_train.mean()),
    "val_positive_rate": float(y_val.mean()),
    "test_positive_rate": float(y_test.mean()),
})


In [ ]:
pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]
)
pipeline.fit(X_train, y_train)

val_prob = pipeline.predict_proba(X_val)[:, 1]
thresholds = [0.3, 0.5, 0.7]
threshold_metrics = []
for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)
    threshold_metrics.append(
        {
            "threshold": threshold,
            "accuracy": accuracy_score(y_val, val_pred),
            "precision": precision_score(y_val, val_pred),
            "recall": recall_score(y_val, val_pred),
            "f1": f1_score(y_val, val_pred),
            "roc_auc": roc_auc_score(y_val, val_prob),
        }
    )
threshold_metrics

# TODO:
# 1. Add at least one tuned variant or additional threshold sweep.
# 2. Choose a final threshold using validation evidence.


In [ ]:
final_threshold = 0.5
test_prob = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= final_threshold).astype(int)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "f1": f1_score(y_test, test_pred),
    "roc_auc": roc_auc_score(y_test, test_prob),
}
test_metrics

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, test_pred, ax=ax)
ax.set_title(f"Confusion matrix at threshold = {final_threshold}")
plt.tight_layout()


## Final analysis prompts

Answer these in complete sentences:

1. Which threshold would you choose, and why?
2. How did the threshold change precision and recall?
3. What kinds of mistakes does the model make?
4. In what application setting would you prefer fewer false negatives or fewer false positives?
